## Init Script for environment Setup

In [0]:

# 2) INSTALL OCR PACKAGES ON EVERY NODE

init_sh = r"""
#!/bin/bash
set -e
echo "Installing OCR toolchain..."
apt-get update -qq
apt-get install -y --no-install-recommends tesseract-ocr
pip install --no-cache-dir opencv-python pillow pytesseract
echo "Done OCR install"
"""

# ensure the init folder exists
dbutils.fs.mkdirs("dbfs:/Workspace/init-scripts")
# overwrite any existing script
dbutils.fs.put("dbfs:/Workspace/init-scripts/install_ocr.sh", init_sh, overwrite=True)

# Then go into your Cluster Configuration → Advanced Options → Init Scripts
# and point to dbfs:/databricks/init/install_ocr.sh
print("Init script written; restart your cluster to take effect.")

Wrote 215 bytes.
Init script written; restart your cluster to take effect.


## Spark Processing

In [0]:
import subprocess

def check_env(_):
    import cv2, pytesseract, subprocess as sp
    # gather versions
    py_opencv   = cv2.__version__
    py_pytess   = pytesseract.get_tesseract_version()  # e.g. “4.1.1”
    tess_bin    = sp.check_output(["tesseract","--version"]).decode().split("\n")[0]
    return f"opencv={py_opencv}  pytesseract={py_pytess}  tesseract_bin={tess_bin}"

# parallelize onto, say, 4 tasks (or more)
results = (spark
    .sparkContext
    .parallelize(range(4), 4)
    .map(check_env)
    .collect()
)

print("\n".join(results))

opencv=4.11.0  pytesseract=4.1.1  tesseract_bin=tesseract 4.1.1
opencv=4.11.0  pytesseract=4.1.1  tesseract_bin=tesseract 4.1.1
opencv=4.11.0  pytesseract=4.1.1  tesseract_bin=tesseract 4.1.1
opencv=4.11.0  pytesseract=4.1.1  tesseract_bin=tesseract 4.1.1


In [0]:

# 3) DEFINE OCR PROCESSING FUNCTION

def process_frame_batch(frame_paths):
    import cv2, pytesseract, os
    from datetime import datetime

    x1, y1, x2, y2 = 465, 30, 770, 65
    results = []

    for frame_path in frame_paths:
        frame_name = os.path.basename(frame_path)
        try:
            file_path = frame_path.replace("dbfs:", "/dbfs")
            img = cv2.imread(file_path)
            if img is None:
                results.append((frame_name, "", f"Cannot read {file_path}"))
                continue

            roi = img[y1:y2, x1:x2]
            gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
            _, thresh = cv2.threshold(gray, 128, 255,
                                     cv2.THRESH_BINARY | cv2.THRESH_OTSU)

            raw = pytesseract.image_to_string(thresh, config="--psm 6")
            txt = raw.strip().replace(".", "").replace(" ", "")

            try:
                dt = datetime.strptime(txt, "%d/%m/%Y%H:%M:%S")
                ts = dt.strftime("%d/%m/%Y%H:%M:%S")
                results.append((frame_name, ts, "Success"))
            except Exception as pe:
                results.append((frame_name, "", f"ParseError({txt}): {pe}"))

        except Exception as e:
            results.append((frame_name, "", f"OCRerror: {e}"))
        finally:
            del img

    return results


# 4) SPARK CONFIG & PARALLEL PROCESSING

from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType

spark.conf.set("spark.sql.adaptive.enabled", True)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", True)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", True)

root = "xxx/Eem/Decoded Frames/Eem_ch04_0619_060343_235956"
batch_dir = f"{root}/metadata_batches"

# discover subdirectories
subdirs = [f.path for f in dbutils.fs.ls(root) if f.isDir()]
subdirs.sort()
print(f"Found {len(subdirs)} subdirectories under {root}")

batch_size = 10
num_batches = (len(subdirs) + batch_size - 1) // batch_size

schema = StructType([
    StructField("frame_name", StringType(), True),
    StructField("actual_timestamp", StringType(), True),
    StructField("status", StringType(), True)
])

for i in range(num_batches):
    lo = i * batch_size
    hi = min(lo + batch_size, len(subdirs))
    this_batch = subdirs[lo:hi]
    print(f"--- Batch {i+1}/{num_batches}: folders {lo+1}–{hi}")

    # collect frame paths
    frame_paths = []
    for d in this_batch:
        try:
            frame_paths += [f.path for f in dbutils.fs.ls(d) if f.path.endswith(".jpg")]
        except Exception as e:
            print(f"  ls error on {d}: {e}")

    if not frame_paths:
        continue

    # how many Spark partitions? ~100 frames each, but at least 32 total
    npart = max(32, len(frame_paths) // 100)
    rdd = spark.sparkContext.parallelize(frame_paths, npart)

    def proc_part(iterable):
        batch = []
        for path in iterable:
            batch.append(path)
            if len(batch) >= 10:
                yield from process_frame_batch(batch)
                batch.clear()
        if batch:
            yield from process_frame_batch(batch)

    res_rdd = rdd.mapPartitions(proc_part)
    df = spark.createDataFrame(res_rdd, schema)

    out_path = f"{batch_dir}/batch_{i:03d}"
    df.write.mode("overwrite").parquet(out_path)

    # avoid an extra Spark job: log input count instead of df.count()
    n_input = len(frame_paths)
    print(f"  → Batch {i:03d} written to {out_path} ({n_input} frames)")

    df.unpersist()
    spark.catalog.clearCache()


# 5) COMBINE & FINAL WRITE

print("Combining all batches...")
combined = spark.read.parquet(f"{batch_dir}/batch_*")
combined = combined.orderBy("frame_name")

metadata_root = "xxx/Eem/metadata" # Output Root for the metadata

combined.coalesce(4) \
        .write.mode("overwrite") \
        .option("header", "true") \
        .csv(f"{metadata_root}/metadata.csv")

combined.write.mode("overwrite") \
        .parquet(f"{metadata_root}/metadata.parquet")

print("Final outputs:")
print(f" - CSV:     {metadata_root}/metadata_Eem_ch04_0619_060343_235956.csv")
print(f" - Parquet: {metadata_root}/metadata_Eem_ch04_0619_060343_235956.parquet")

# 6) CLEANUP INTERMEDIATE FILES

print("Deleting intermediate batch files...")
dbutils.fs.rm(batch_dir, recurse=True)
print(f"Deleted: {batch_dir}")